# 03 — Fine-tuning

RLDX-1 has a single unified training entry point —
`rldx/experiment/launch_train.py` — that covers pre-training,
mid-training, and fine-tuning. This notebook walks you through:

1. The CLI surface (everything is a `TrainConfig` field).
2. The core flags you always need.
3. A one-step dry run so you can confirm your dataset, modality
   config, and base checkpoint all line up before committing to a
   long job.
4. The layout of the saved checkpoint.

This notebook spawns a subprocess that uses the GPU. A single GPU
is enough for the one-step dry run.


## 1. Core flags cheat-sheet

| Flag | Purpose |
|---|---|
| `--base-model-path` | HF repo id or local path of the checkpoint to start from. |
| `--dataset-path` | Path to the LeRobot dataset root. Use `--dataset-paths` + `--dataset-mix-ratios` for mixtures. |
| `--embodiment-tag` | `GENERAL_EMBODIMENT` for an unseen robot; an existing enum value otherwise. |
| `--modality-config-path` | Python file defining the modality config dict (see notebook 02). |
| `--output-dir` | Where to save checkpoints. |
| `--max-steps` | Number of optimizer steps. |
| `--global-batch-size` | Effective batch size *across all GPUs*. |
| `--learning-rate` | Defaults to `1e-4`. |
| `--save-steps` / `--save-total-limit` | Checkpoint cadence and retention. |
| `--video-length` / `--video-stride` | Video history length & stride. Defaults: `4` / `2`. |
| `--color-jitter-params` | `brightness X contrast Y saturation Z hue W` augmentation. |
| `--use-wandb` / `--wandb-project` / `--experiment-name` | W&B logging. |

Every field of the `TrainConfig` dataclass (in
`rldx/configs/train_config.py`) is exposed as a CLI flag automatically
via `tyro` — dashes replace underscores. `--help` on the launcher
prints the full list.


## 2. Inspecting `TrainConfig` from Python

Sometimes it's quicker to introspect the dataclass than to grep the
help text. The fields below are a stable subset — the whole dataclass
is ~200 fields across pre-training, fine-tuning, memory, motion, physics.


In [ ]:
import dataclasses
from rldx.configs.train_config import TrainConfig

stable = {
    'base_model_path', 'dataset_path', 'modality_config_path',
    'embodiment_tag', 'output_dir', 'max_steps', 'global_batch_size',
    'learning_rate', 'save_steps', 'save_total_limit',
    'video_length', 'video_stride', 'color_jitter_params', 'n_cog_tokens',
    'use_wandb', 'wandb_project', 'experiment_name',
}
for f in dataclasses.fields(TrainConfig):
    if f.name in stable:
        default = f.default if f.default is not dataclasses.MISSING else '<required>'
        print(f'  --{f.name.replace("_", "-"):28s} = {default!r}')


## 3. A typical fine-tune command

The following is the canonical robocasa fine-tune invocation, with
the flags spelled out. Run it from a shell (not from this notebook)
when you are ready to commit to a real training run.

```bash
uv run torchrun --nproc_per_node=8 --master_port=29500 \
    rldx/experiment/launch_train.py \
        --base-model-path RLWRLD/RLDX-1-PT \
        --dataset-path /path/to/lerobot_dataset \
        --embodiment-tag GENERAL_EMBODIMENT \
        --modality-config-path rldx/configs/data/robocasa_config.py \
        --output-dir ckpt/rldx_ft_robocasa \
        --video-length 4 \
        --n-cog-tokens 64 \
        --color-jitter-params brightness 0.3 contrast 0.4 saturation 0.5 hue 0.08 \
        --global-batch-size 64 \
        --learning-rate 1e-4 \
        --max-steps 60000 \
        --save-steps 1000 --save-total-limit 5 \
        --dataloader-num-workers 12 \
        --use-wandb --wandb-project rldx-finetune \
        --experiment-name rldx_ft_robocasa_60k
```

The example above uses `RLWRLD/RLDX-1-PT` as the fine-tune base.


## 4. One-step validation run

Before committing to a 60k-step run, kick off a one-step job
against a single GPU. This verifies that:

- The base checkpoint downloads / loads.
- Your dataset is readable and the modality config matches.
- Normalization statistics compute successfully.
- One forward + backward + optimizer step completes without OOM.

Set `BASE_MODEL`, `DATA_PATH`, and `MODALITY_PATH` below, then run
the cell. It spawns `launch_train.py` as a subprocess with
`--max-steps 1 --save-steps 1 --global-batch-size 1` against a
single GPU.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path.cwd().parent  # notebook lives in getting_started/; repo root is one level up
BASE_MODEL    = 'RLWRLD/RLDX-1-PT'
DATA_PATH     = '/path/to/your/lerobot_dataset'
MODALITY_PATH = REPO / 'rldx/configs/data/robocasa_config.py'
OUTPUT_DIR    = REPO / 'ckpt/_gs_one_step'

cmd = [
    'uv', 'run', 'torchrun',
    '--nproc_per_node=1', '--master_port=29511',
    str(REPO / 'rldx/experiment/launch_train.py'),
    '--base-model-path', BASE_MODEL,
    '--dataset-path', DATA_PATH,
    '--embodiment-tag', 'GENERAL_EMBODIMENT',
    '--modality-config-path', str(MODALITY_PATH),
    '--output-dir', str(OUTPUT_DIR),
    '--video-length', '4',
    '--n-cog-tokens', '64',
    '--global-batch-size', '1',
    '--max-steps', '1',
    '--save-steps', '1',
    '--dataloader-num-workers', '0',
    # `use_wandb` defaults to False, so we omit --use-wandb here.
]

print('>>>', ' '.join(cmd))
# Uncomment to actually run. Expect ~3-5 minutes on a single GPU the
# first time (base checkpoint download dominates).
# result = subprocess.run(cmd, cwd=REPO, check=False)
# print('exit code:', result.returncode)


## 5. Checkpoint layout

After a successful run, `OUTPUT_DIR/checkpoint-<step>/` contains:

```
checkpoint-1/
├── config.json                  # HF AutoConfig — model_type=RLDX-1
├── model.safetensors(.index.json)
├── experiment_cfg/
│   ├── config.yaml              # full run_config snapshot (loaded by launch_train)
│   ├── dataset_statistics.json
│   └── final_model_config.json
├── processor/
│   ├── processor_config.json    # RLDXProcessor kwargs
│   ├── statistics.json          # per-key normalization stats
│   └── embodiment_id.json
└── trainer_state.json           # HF Trainer state for resume
```

Pass the `checkpoint-N/` path back as `--base-model-path` to resume,
or as `model_path=` to `RLDXPolicy(...)` for inference.


In [ ]:
from pathlib import Path

ckpt_root = Path('/path/to/your/output_dir')  # set me
if ckpt_root.exists():
    for p in sorted(ckpt_root.rglob('*')):
        if p.is_file() and p.stat().st_size > 0:
            rel = p.relative_to(ckpt_root)
            print(f'{rel}  ({p.stat().st_size // 1024} KiB)')
else:
    print('(skip — set ckpt_root to a real output directory)')


## 6. Next

- Serve your fine-tuned checkpoint over a network socket →
  [`04_inference_server.ipynb`](04_inference_server.ipynb)
- Benchmark it in simulation → [`../docs/evaluation.md`](../docs/evaluation.md)
